# CE-SSL training audio transforms demo

This notebook runs the **same matched transform stack** as `MatchedSpeechInNoiseDatasetBatched` (used by `LitAudioSSL` in `lightning_ssl_matched_speech_in_noise.py`). Hyperparameters come from the CE-SSL config (`kell2018_barlow_equivariant`, eq λ=0.5).

**Notebook flow:** load config → build transform objects → load a fixed real speech/noise batch → apply the matched pipeline once to get four mixtures (**11 / 12 / 21 / 22**) → plot waveforms and inspect crop/augment in isolation.

View names encode `(speech clip, noise clip)` — e.g. `combined_12` is speech clip 1 with noise clip 2. Two independent `matched_random_crop` + optional augment + mix passes yield the four views used by the paired Barlow loss.

**Cluster vs notebook:** HDF5 extraction is compute/I/O heavy — regenerate `demo_audio_batch/` on the cluster with `sbatch slurm_scripts/extract_demo_audio_batch.sh` (not locally). This notebook only loads the bundled `.npy` clips (or pre-extracted assets in git) and applies transforms in memory.

In [ ]:
from pathlib import Path
import json
import logging
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from IPython.display import Audio, display

# Resolve repo root whether the notebook is run from repo root, notebooks/, or demo_notebooks/
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "demo_notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
elif PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import robustness.audio_functions.audio_transforms as at

# Load the release CE-SSL config (SNR bounds, dBSPL target, augment flags)
CONFIG_PATH = PROJECT_ROOT / "model_configs/kell2018_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

at_cfg = config["audio_transforms"]
data_cfg = config["data"]
SR = 20_000
SIG_LEN = 40_000  # 2 s at 20 kHz — same clip length as the training dataloader

logging.getLogger("sox").setLevel(logging.ERROR)

# Transform objects — same construction as MatchedSpeechInNoiseDatasetBatched.__init__
random_crop_noise = at.RandomCrop(SIG_LEN)  # independent crop for each noise clip
matched_random_crop = at.MatchedRandomSignalCrops(  # word-center-aligned crops from two speech clips
    SIG_LEN, skip_aug_match=data_cfg.get("skip_aug_match", False)
)
matched_combiner = at.MatchedCombineWithRandomDBSNR(  # same SNR within each (speech, noise) pair
    at_cfg["low_snr"], at_cfg["high_snr"], return_param=True
)
set_dbSPL = at.DBSPLNormalizeForegroundAndBackground(at_cfg["dbspl"])
matched_signal_augment = None
if data_cfg.get("signal_augment", False):
    matched_signal_augment = at.MatchedRandomSignalAugmentSox(  # shared Sox params within each pair
        sample_rate=SR, skip_aug_match=data_cfg.get("skip_aug_match", False)
    )

print("Training transform settings:")
print(f"  SNR range: [{at_cfg['low_snr']}, {at_cfg['high_snr']}] dB")
print(f"  target SPL: {at_cfg['dbspl']} dB")
print(f"  signal_augment: {data_cfg.get('signal_augment', False)}")

DEMO_AUDIO_DIR = PROJECT_ROOT / "notebooks/demo_notebooks/demo_audio_batch"
EXTRACT_SLURM = "slurm_scripts/extract_demo_audio_batch.sh"


def load_demo_batch(demo_dir=DEMO_AUDIO_DIR):
    """Load one pre-extracted training batch element (pre-transform waveforms)."""
    demo_dir = Path(demo_dir)
    missing = [demo_dir / f"{name}.npy" for name in ("speech_1", "speech_2", "noise_1", "noise_2") if not (demo_dir / f"{name}.npy").exists()]
    if missing:
        raise FileNotFoundError(
            "Demo audio batch not found. Set COCHDNN_JSIN_TRAIN_H5 and "
            "COCHDNN_AUDIONOISE_TRAIN_H5, then submit from the repo root:\n"
            f"  sbatch {EXTRACT_SLURM}\n"
            f"Missing: {', '.join(p.name for p in missing)}"
        )
    batch = {name: np.load(demo_dir / f"{name}.npy") for name in ("speech_1", "speech_2", "noise_1", "noise_2")}
    meta_path = demo_dir / "metadata.json"
    if meta_path.exists():
        with open(meta_path) as f:
            batch["metadata"] = json.load(f)
    return batch


def show_waveforms(items, n=1200):
    """Plot the first n samples of each waveform for quick visual comparison."""
    plt.figure(figsize=(10, 2.5 * len(items)))
    for i, (name, wav) in enumerate(items, 1):
        wav = torch.as_tensor(wav).detach().cpu().float()
        plt.subplot(len(items), 1, i)
        plt.plot(wav[:n])
        plt.title(name)
    plt.tight_layout()

In [ ]:
def apply_ce_ssl_training_pair(speech_1, speech_2, noise_1, noise_2):
    """Mirror one MatchedSpeechInNoiseDatasetBatched element: four views (11, 12, 21, 22)."""
    # Training dataset always treats speech_1 as the shorter clip
    if len(speech_1) > len(speech_2):
        speech_1, speech_2 = speech_2, speech_1

    # Crop each noise clip independently to SIG_LEN
    noise_1 = torch.tensor(random_crop_noise(noise_1))
    noise_2 = torch.tensor(random_crop_noise(noise_2))

    # Pass 1 → views 11 and 21 (shared crop/augment params within the pair)
    cropped_11, cropped_21 = matched_random_crop(speech_1, speech_2)
    # Pass 2 → views 12 and 22 (independent random crop/augment)
    cropped_12, cropped_22 = matched_random_crop(speech_1, speech_2)

    if matched_signal_augment is not None:
        cropped_11, cropped_21 = matched_signal_augment(cropped_11, cropped_21)
        cropped_12, cropped_22 = matched_signal_augment(cropped_12, cropped_22)

    cropped_11, cropped_21 = torch.tensor(cropped_11), torch.tensor(cropped_21)
    cropped_12, cropped_22 = torch.tensor(cropped_12), torch.tensor(cropped_22)

    # Mix speech with noise: view 1 uses noise_1 for both clips; view 2 uses noise_2
    combined_11, combined_21, snr_1 = matched_combiner(cropped_11, cropped_21, noise_1, noise_1)
    combined_12, combined_22, snr_2 = matched_combiner(cropped_12, cropped_22, noise_2, noise_2)

    # Normalize each mixture to the config target dBSPL
    combined_11, _ = set_dbSPL(combined_11, None)
    combined_12, _ = set_dbSPL(combined_12, None)
    combined_21, _ = set_dbSPL(combined_21, None)
    combined_22, _ = set_dbSPL(combined_22, None)

    return {
        "combined_11": combined_11,
        "combined_12": combined_12,
        "combined_21": combined_21,
        "combined_22": combined_22,
        "snr_1": snr_1,
        "snr_2": snr_2,
    }


# Fixed real JSIN/Audionoise clips (pre-transform); see demo_audio_batch/metadata.json
demo = load_demo_batch()
speech_1 = demo["speech_1"]
speech_2 = demo["speech_2"]
noise_1 = demo["noise_1"]
noise_2 = demo["noise_2"]
if "metadata" in demo:
    print("Loaded demo batch:", demo["metadata"].get("lengths_samples"))

views = apply_ce_ssl_training_pair(speech_1, speech_2, noise_1, noise_2)
print(f"Mix SNR view-1 pair: {views['snr_1']:.2f} dB, view-2 pair: {views['snr_2']:.2f} dB")

In [ ]:
# Plot inputs, cropped noise, and all four CE-SSL mixture views
show_waveforms([
    (f"speech_1 ({len(speech_1) / SR:.2f} s)", speech_1),
    (f"speech_2 ({len(speech_2) / SR:.2f} s)", speech_2),
    ("noise_1 crop", noise_1[:SIG_LEN] if len(noise_1) >= SIG_LEN else noise_1),
    ("combined_11", views["combined_11"]),
    ("combined_12", views["combined_12"]),
    ("combined_21", views["combined_21"]),
    ("combined_22", views["combined_22"]),
])
plt.show()
display(Audio(views["combined_11"].numpy(), rate=SR))  # listen to one 2 s mixture

In [ ]:
# --- Isolated look at matched crop and augment (one pass of the pipeline) ---

# MatchedRandomSignalCrops: same random offset relative to word center on both clips
c11, c21 = matched_random_crop(speech_1, speech_2)
print(f"Matched 2 s crops: {len(c11)} samples each")

# MatchedRandomSignalAugmentSox: identical Sox chain on both clips in the pair
if matched_signal_augment is not None:
    a11, a21 = matched_signal_augment(c11, c21, print_augments=True)
    show_waveforms([
        ("speech_1", speech_1),
        ("speech_2", speech_2),
        ("matched crop speech_1", c11),
        ("matched augment speech_1", a11),
        ("matched augment speech_2", a21),
    ])
    plt.show()
else:
    print("signal_augment is disabled in config")

## How this connects to CE-SSL training

`MatchedSpeechInNoiseDatasetBatched` returns four mixtures per batch element (`combined_11`, `combined_12`, `combined_21`, `combined_22`). `LitAudioSSL.collate_fn` stacks them for the encoder; the **paired Barlow loss** (`Paired_Loss` in the config) treats views that share matched crop/augment/SNR parameters as equivariant pairs (e.g. 11↔21 share pass-1 params; 12↔22 share pass-2 params).

**Source files:** `lightning_scripts/jsinV3DataLoader_precombined_batched.py` (dataset + collate) and `robustness/audio_functions/audio_transforms.py` (`MatchedRandomSignalCrops`, `MatchedRandomSignalAugmentSox`, `MatchedCombineWithRandomDBSNR`).